In [1]:
#Inspect if the outputs from the 10 random runs deviate significantly from each other. 
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from netCDF4 import Dataset
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.metrics import r2_score as R2
from sklearn.model_selection import KFold
from copy import deepcopy
import utils
from unet import UNet_nobatchnorm
from scipy.stats import pearsonr
from pathlib import Path
import numpy.fft as fft
from matplotlib.colors import TwoSlopeNorm
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import helper_functions as hf

In [2]:
root_dir = '/work/uo0780/u241359/project_tide_synergy/data/'
nctrains, nctest = hf.load_data_from_nc_as_lists(root_dir)

Ntrain_fromfiles = np.sum([nc.dimensions['time_counter'].size for nc in nctrains], axis = 0)
Ntest_fromfiles = np.sum([nc.dimensions['time_counter'].size for nc in nctest], axis = 0)

model_folder = '/work/uo0780/u241359/project_tide_synergy/trainedmodels_forpaper/'
#filesuffix1 + randomseednumber + filesuffix2 = file suffix. 
filesuffix1='_ssh_cosssh_sin_nobatchnorm_rp_'
filesuffix2='.pth'

vel_cmap  = 'BrBG' #'viridis'
vort_cmap = 'PRGn'
ssh_cmap  = 'bwr'
sst_cmap = 'inferno'

bottom_slice = slice(0,256)
mid_slice = slice(232, 488)
top_slice = slice(464, 720)

def corr(data, mod):
    return pearsonr(data.flatten(), mod.flatten())[0]
def L2_R(data,mod):
    return R2(data.flatten(), mod.flatten())

nensemble = 10 #How many re-trained U-Net models there are

In [3]:
Nbase = 16
def totorch(x):
    return torch.tensor(x, dtype = torch.float).cpu()
    
def preload_data(nctrains, total_records):
    #total_records = Ntrain#sum(nc.dimensions['time_counter'].size for nc in nctrains)
    #dimensions of data of the nc file.
    max_height = 722
    max_width = 258
    all_input_data = np.zeros((total_records, N_inp, max_height, max_width))*np.nan
    all_output_data = np.zeros((total_records, N_out, max_height, max_width))*np.nan
    current_index = 0
    for ncindex, ncdata in enumerate(nctrains):
        num_recs = ncdata.dimensions['time_counter'].size
        rec_slice = slice(current_index, current_index + num_recs)
        
        for ind, var_name in enumerate(var_input_names):
            data_slice = np.squeeze(ncdata.variables[var_name])
            # print('data_slice shape:')
            # print(data_slice.shape)        
            #all_input_data[rec_slice, ind, :, :] = data_slice
            #For some variables, the dimensions in (x, y) may be smaller than (max_height, max_width). Changing the code so that it adapts them.
            # Get the actual dimensions of data_slice
            slice_height, slice_width = data_slice.shape[-2], data_slice.shape[-1]
            # Place data_slice into the corresponding slice of all_input_data
            all_input_data[rec_slice, ind, :slice_height, :slice_width] = data_slice
    

        for ind, var_name in enumerate(var_output_names):
            data_slice = np.squeeze(ncdata.variables[var_name])
            #all_output_data[rec_slice, ind, :, :] = data_slice
            # Get the actual dimensions of data_slice
            slice_height, slice_width = data_slice.shape[-2], data_slice.shape[-1]
            # Place data_slice into the corresponding slice of all_input_data
            all_output_data[rec_slice, ind, :slice_height, :slice_width] = data_slice

        current_index += num_recs
        
    return all_input_data, all_output_data

# # Modify the loadtrain function to pull data from preloaded memory
# def loaddata_preloaded_train(index, batch_size, all_input_data, all_output_data):
#     rec_slice = slice(index, index + batch_size)
#     lim = 720
#     width = 256
#     yslice = slice(0, lim)
#     xslice = slice(0, width)
#     # print('rec_slice is:')
#     # print(rec_slice)
#     # print('mean of squared values of loaded input data:')
#     # print("{0:0.32f}".format(np.nanmean(all_input_data[rec_slice, :, yslice, xslice]**2)))
#     return (all_input_data[rec_slice, :, yslice, xslice], 
#             all_output_data[rec_slice, :, yslice, xslice])
#Load test data as one single batch
def loaddata_preloaded_test(all_input_data, all_output_data):
    #rec_slice = slice(index, index + batch_size)
    lim = 720
    width = 256
    yslice = slice(0, lim)
    xslice = slice(0, width)
    # print('rec_slice is:')
    # print(rec_slice)
    # print('mean of squared values of loaded input data:')
    # print("{0:0.32f}".format(np.nanmean(all_input_data[rec_slice, :, yslice, xslice]**2)))
    return (all_input_data[:, :, yslice, xslice], 
            all_output_data[:, :, yslice, xslice])


def load_variable(ncdata, ncindex, variable, rec_slice, yslice, xslice):
    data_squeezed = np.squeeze(ncdata[ncindex].variables[variable])
    return data_squeezed[rec_slice, yslice, xslice]
def apply_fixed_offset_ssh_leads_by_segments(all_input, all_output, shifted_input_indices, seg_lengths, lag=1):
    """
    Apply a +lag time shift to selected input channels *within each segment* (each .nc file),
    dropping the last 'lag' snapshots of each segment. No cross-file pairing.

    SSH leads => keep SSH channel(s) aligned at time t,
                shift other channels so x_shifted[t] = original x_shifted[t+lag].
    
    all_input:  (T, Cin, H, W)
    all_output: (T, Cout, H, W)
    seg_lengths: list like [150,150,150,150] for 4 training sims
    """
    assert lag == 1, "Written for lag=1 (easy to generalize later)."
    shifted_input_indices = list(shifted_input_indices)

    T, Cin, H, W = all_input.shape
    Tout, Cout = all_output.shape[0], all_output.shape[1]
    assert Tout == T, "Input/output time dimension mismatch."

    # total kept samples = sum(L - lag) over segments
    new_T = sum(L - lag for L in seg_lengths)
    x_new = np.zeros((new_T, Cin, H, W), dtype=all_input.dtype)
    y_new = np.zeros((new_T, Cout, H, W), dtype=all_output.dtype)

    in_pos = 0
    out_pos = 0

    for L in seg_lengths:
        if L <= lag:
            raise ValueError(f"Segment length {L} too short for lag={lag}.")

        s0 = in_pos
        s1 = in_pos + L
        keep = L - lag

        # Start by copying the "base" time-aligned inputs for all channels: t = s0 ... s1-lag-1
        x_new[out_pos:out_pos+keep, :, :, :] = all_input[s0:s1-lag, :, :, :]

        # Now overwrite ONLY shifted channels with t+lag values: t = s0+lag ... s1-1
        x_new[out_pos:out_pos+keep, shifted_input_indices, :, :] = all_input[s0+lag:s1, shifted_input_indices, :, :]

        # Outputs remain aligned with base time (SSH time): drop last lag
        y_new[out_pos:out_pos+keep, :, :, :] = all_output[s0:s1-lag, :, :, :]

        in_pos += L
        out_pos += keep

    return x_new, y_new

def pad_to_multiple_by_repeating_last(all_input, all_output, batch_size):
    """
    Pads the *end* of the dataset by repeating the last snapshot. This is so that the number of snapshots remain unchanged after calling apply_fixed_offset_ssh_leads. 
    The reason why I want to do that is because I want the number of snapshots to be divisible by batch sizes during training. It artifically repeats the last snapshot, 
    but that is fine, as that is just one snapshot each simulation 150 snapshots).
    """
    N = all_input.shape[0]
    r = N % batch_size #If r == 0, then N is already an exact multiple of batch_size. In that case, you don’t need any padding, so the function just returns the arrays unchanged:
    if r == 0:
        return all_input, all_output

    n_pad = batch_size - r
    inp_pad = np.repeat(all_input[-1:, ...], n_pad, axis=0)
    out_pad = np.repeat(all_output[-1:, ...], n_pad, axis=0)

    all_input  = np.concatenate([all_input,  inp_pad], axis=0)
    all_output = np.concatenate([all_output, out_pad], axis=0)
    return all_input, all_output

    


In [4]:
# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓ Change below for each Configuration ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓
vi1 = 'ssh_ins'
vi2 = 'u_xy_ins'
vi3 = 'v_xy_ins'

vo1 = 'ssh_cos'
vo2 = 'ssh_sin'

save_fn_prefix  = 'any_{}{}{}_{}{}_nobatchnorm_offset_48hr_ssh_leads'.format(vi1, vi2, vi3, vo1, vo2)

batch_size = 60 #maximizing it so that the GPU memory maxes out. Needs to be divisible by Ntrain. Otherwise there will be size mismatch issues.

var_input_names = [vi1, vi2, vi3]
var_output_names = [vo1, vo2]
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑
N_inp = len(var_input_names)
N_out = len(var_output_names)

nctrains, nctest = hf.load_data_from_nc_as_lists(root_dir)

# Add offset, using the ordering in var_input_names:
# var_input_names = ['ssh_ins','u_xy_ins','v_xy_ins']
# segment lengths per .nc file (train: 4 sims, test: 1 sim)
seg_train = [nc.dimensions['time_counter'].size for nc in nctrains]
seg_test  = [nc.dimensions['time_counter'].size for nc in nctest]

all_train_input, all_train_output = preload_data(nctrains, Ntrain_fromfiles)
all_test_input,  all_test_output  = preload_data(nctest,  Ntest_fromfiles)

# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓ Change below for each Configuration ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

# SSH leads by 1 snapshot => shift U and V channels forward by 1 within each file
# var_input_names = ['ssh_ins','u_xy_ins','v_xy_ins']  => shift indices [1,2]
all_train_input, all_train_output = apply_fixed_offset_ssh_leads_by_segments(
    all_train_input, all_train_output, shifted_input_indices=[1,2], seg_lengths=seg_train, lag=1
)
all_test_input, all_test_output = apply_fixed_offset_ssh_leads_by_segments(
    all_test_input, all_test_output, shifted_input_indices=[1,2], seg_lengths=seg_test, lag=1
)
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑


#Normalize data
#Compute mean and variance for normalization
mean_input=np.nanmean(np.concatenate((all_train_input, all_test_input), axis=0),axis=(0, 2, 3))
mean_output=np.nanmean(np.concatenate((all_train_output, all_test_output), axis=0),axis=(0, 2, 3))
#Subtract the data with their means
all_train_input=all_train_input-mean_input[None, :, None, None]
all_train_output=all_train_output-mean_output[None, :, None, None]
all_test_input=all_test_input-mean_input[None, :, None, None]
all_test_output=all_test_output-mean_output[None, :, None, None]
#Compute the variances
var_input=np.nanmean((np.concatenate((all_train_input, all_test_input), axis=0))**2,axis=(0, 2, 3))
var_output=np.nanmean((np.concatenate((all_train_output, all_test_output), axis=0))**2,axis=(0, 2, 3))
print("mean and variance of all input data:")
print(mean_input,var_input)
print("mean and variance of all output data:")
print(mean_output,var_output)
#Scale the data so that they have variance of 1
all_train_input=all_train_input/np.sqrt(var_input[None, :, None, None])
all_train_output=all_train_output/np.sqrt(var_output[None, :, None, None])
all_test_input=all_test_input/np.sqrt(var_input[None, :, None, None])
all_test_output=all_test_output/np.sqrt(var_output[None, :, None, None])
#Have checked that after these operations, the data is scaled to be zero mean and variance 1.

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and before padding):', Ntrain)
print('number of testing records (after offset):', Ntest)
print('train input shape:')
print(all_train_input.shape)


#Pad so that the number of training data is divisible by batch 
all_train_input, all_train_output = pad_to_multiple_by_repeating_last(
    all_train_input, all_train_output, batch_size)

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and padding):', Ntrain)
print('number of testing records (after offset):', Ntest)


inp_test, out_test = loaddata_preloaded_test(all_test_input, all_test_output)
out_test = out_test*np.sqrt(var_output[None, :, None, None])+mean_output[None, :, None, None]

truth_bot = out_test[:, :, bottom_slice, :]
truth_mid = out_test[:, :, mid_slice, :]
truth_top = out_test[:, :, top_slice, :]

combined_names = ''.join(var_input_names)

#Array to record performance metrics
corr_ensemble_full = np.zeros(nensemble)
R2_ensemble_full = np.zeros(nensemble)
corr_ensemble_top = np.zeros(nensemble)
R2_ensemble_top = np.zeros(nensemble)
corr_ensemble_mid = np.zeros(nensemble)
R2_ensemble_mid = np.zeros(nensemble)
corr_ensemble_bot = np.zeros(nensemble)
R2_ensemble_bot = np.zeros(nensemble)
for iensemble in np.arange(nensemble):
    fstr = f'{save_fn_prefix}_rp_{iensemble}' 
    model_filename = f'/{fstr}.pth'
    model_path = model_folder+ model_filename
    state_dict = torch.load(model_path)


    # Create a new instance of the model
    model = UNet_nobatchnorm(N_inp, N_out, bilinear = True, Nbase = Nbase)
    # Load the state_dict into the model
    model.load_state_dict(state_dict)
    # Set the model to evaluation mode
    model.eval()
    
    with torch.no_grad():
        out_mod = model(totorch(inp_test)).detach().cpu().numpy()
    #Renormalize
    out_mod = out_mod*np.sqrt(var_output[None, :, None, None])+mean_output[None, :, None, None]

    mod_bot = out_mod[:, :, bottom_slice, :]
    mod_mid = out_mod[:, :, mid_slice, :]
    mod_top = out_mod[:, :, top_slice, :]

    corr_ensemble_full[iensemble] = corr(out_test, out_mod)
    R2_ensemble_full[iensemble] = L2_R(out_test, out_mod)    
    corr_ensemble_top[iensemble] = corr(truth_top, mod_top)
    R2_ensemble_top[iensemble] = L2_R(truth_top, mod_top)  
    corr_ensemble_mid[iensemble] = corr(truth_mid, mod_mid)
    R2_ensemble_mid[iensemble] = L2_R(truth_mid, mod_mid)
    corr_ensemble_bot[iensemble] = corr(truth_bot, mod_bot)
    R2_ensemble_bot[iensemble] = L2_R(truth_bot, mod_bot)

print('full panel, correlation:')
mean_val = np.mean(corr_ensemble_full)
max_val = np.max(corr_ensemble_full)
min_val = np.min(corr_ensemble_full)
std_val = np.std(corr_ensemble_full)
latex_table_row=f', full & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('full panel, R2:')
mean_val = np.mean(R2_ensemble_full)
max_val = np.max(R2_ensemble_full)
min_val = np.min(R2_ensemble_full)
std_val = np.std(R2_ensemble_full)
latex_table_row=f', full & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('midjet panel, correlation:')
mean_val = np.mean(corr_ensemble_mid)
max_val = np.max(corr_ensemble_mid)
min_val = np.min(corr_ensemble_mid)
std_val = np.std(corr_ensemble_mid)
latex_table_row=f', mid & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('midjet panel, R2:')
mean_val = np.mean(R2_ensemble_mid)
max_val = np.max(R2_ensemble_mid)
min_val = np.min(R2_ensemble_mid)
std_val = np.std(R2_ensemble_mid)
latex_table_row=f', mid & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)

mean and variance of all input data:
[ 0.03322287  0.0356514  -0.00192003] [0.31186455 0.04685245 0.0488286 ]
mean and variance of all output data:
[-5.16332561e-04 -9.81347005e-05] [9.36171329e-05 1.01426415e-04]
number of training records (after offset and before padding): 596
number of testing records (after offset): 149
train input shape:
(596, 3, 722, 258)
number of training records (after offset and padding): 600
number of testing records (after offset): 149


/tmp/ipykernel_443280/913079040.py:108: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path)
/tmp/ipykernel_443280/913079040.py:108: FutureWarni

full panel, correlation:
, full & 0.8823 & 0.8853 & 0.8761 & 0.0031 \\
full panel, R2:
, full & 0.7775 & 0.7831 & 0.7669 & 0.0053 \\
midjet panel, correlation:
, mid & 0.8220 & 0.8260 & 0.8146 & 0.0038 \\
midjet panel, R2:
, mid & 0.6736 & 0.6809 & 0.6619 & 0.0060 \\


In [5]:
# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓ Change below for each Configuration ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓
vi1 = 'ssh_ins'
vi2 = 'T_xy_ins'

vo1 = 'ssh_cos'
vo2 = 'ssh_sin'

save_fn_prefix  = 'any_{}{}_{}{}_nobatchnorm_offset_48hr_ssh_leads'.format(vi1, vi2, vo1, vo2)
var_input_names = [vi1, vi2]
var_output_names = [vo1, vo2]

batch_size = 80 #maximizing it so that the GPU memory maxes out. Needs to be divisible by Ntrain. Otherwise there will be size mismatch issues.
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑
N_inp = len(var_input_names)
N_out = len(var_output_names)

nctrains, nctest = hf.load_data_from_nc_as_lists(root_dir)

# Add offset, using the ordering in var_input_names:
# var_input_names = ['ssh_ins','u_xy_ins','v_xy_ins']
# segment lengths per .nc file (train: 4 sims, test: 1 sim)
seg_train = [nc.dimensions['time_counter'].size for nc in nctrains]
seg_test  = [nc.dimensions['time_counter'].size for nc in nctest]

all_train_input, all_train_output = preload_data(nctrains, Ntrain_fromfiles)
all_test_input,  all_test_output  = preload_data(nctest,  Ntest_fromfiles)

# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓ Change below for each Configuration ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

#Add offset
# var_input_names = ['ssh_ins','T_xy_ins'] => shift index [1]
all_train_input, all_train_output = apply_fixed_offset_ssh_leads_by_segments(
    all_train_input, all_train_output, shifted_input_indices=[1], seg_lengths=seg_train, lag=1
)
all_test_input, all_test_output = apply_fixed_offset_ssh_leads_by_segments(
    all_test_input, all_test_output, shifted_input_indices=[1], seg_lengths=seg_test, lag=1
)

# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑


#Normalize data
#Compute mean and variance for normalization
mean_input=np.nanmean(np.concatenate((all_train_input, all_test_input), axis=0),axis=(0, 2, 3))
mean_output=np.nanmean(np.concatenate((all_train_output, all_test_output), axis=0),axis=(0, 2, 3))
#Subtract the data with their means
all_train_input=all_train_input-mean_input[None, :, None, None]
all_train_output=all_train_output-mean_output[None, :, None, None]
all_test_input=all_test_input-mean_input[None, :, None, None]
all_test_output=all_test_output-mean_output[None, :, None, None]
#Compute the variances
var_input=np.nanmean((np.concatenate((all_train_input, all_test_input), axis=0))**2,axis=(0, 2, 3))
var_output=np.nanmean((np.concatenate((all_train_output, all_test_output), axis=0))**2,axis=(0, 2, 3))
print("mean and variance of all input data:")
print(mean_input,var_input)
print("mean and variance of all output data:")
print(mean_output,var_output)
#Scale the data so that they have variance of 1
all_train_input=all_train_input/np.sqrt(var_input[None, :, None, None])
all_train_output=all_train_output/np.sqrt(var_output[None, :, None, None])
all_test_input=all_test_input/np.sqrt(var_input[None, :, None, None])
all_test_output=all_test_output/np.sqrt(var_output[None, :, None, None])
#Have checked that after these operations, the data is scaled to be zero mean and variance 1.

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and before padding):', Ntrain)
print('number of testing records (after offset):', Ntest)
print('train input shape:')
print(all_train_input.shape)


#Pad so that the number of training data is divisible by batch 
all_train_input, all_train_output = pad_to_multiple_by_repeating_last(
    all_train_input, all_train_output, batch_size)

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and padding):', Ntrain)
print('number of testing records (after offset):', Ntest)


inp_test, out_test = loaddata_preloaded_test(all_test_input, all_test_output)
out_test = out_test*np.sqrt(var_output[None, :, None, None])+mean_output[None, :, None, None]

truth_bot = out_test[:, :, bottom_slice, :]
truth_mid = out_test[:, :, mid_slice, :]
truth_top = out_test[:, :, top_slice, :]

combined_names = ''.join(var_input_names)

#Array to record performance metrics
corr_ensemble_full = np.zeros(nensemble)
R2_ensemble_full = np.zeros(nensemble)
corr_ensemble_top = np.zeros(nensemble)
R2_ensemble_top = np.zeros(nensemble)
corr_ensemble_mid = np.zeros(nensemble)
R2_ensemble_mid = np.zeros(nensemble)
corr_ensemble_bot = np.zeros(nensemble)
R2_ensemble_bot = np.zeros(nensemble)
for iensemble in np.arange(nensemble):
    fstr = f'{save_fn_prefix}_rp_{iensemble}' 
    model_filename = f'/{fstr}.pth'
    model_path = model_folder+model_filename
    state_dict = torch.load(model_path)
    # Create a new instance of the model
    model = UNet_nobatchnorm(N_inp, N_out, bilinear = True, Nbase = Nbase)
    # Load the state_dict into the model
    model.load_state_dict(state_dict)
    # Set the model to evaluation mode
    model.eval()
    
    with torch.no_grad():
        out_mod = model(totorch(inp_test)).detach().cpu().numpy()
    #Renormalize
    out_mod = out_mod*np.sqrt(var_output[None, :, None, None])+mean_output[None, :, None, None]

    mod_bot = out_mod[:, :, bottom_slice, :]
    mod_mid = out_mod[:, :, mid_slice, :]
    mod_top = out_mod[:, :, top_slice, :]

    corr_ensemble_full[iensemble] = corr(out_test, out_mod)
    R2_ensemble_full[iensemble] = L2_R(out_test, out_mod)    
    corr_ensemble_top[iensemble] = corr(truth_top, mod_top)
    R2_ensemble_top[iensemble] = L2_R(truth_top, mod_top)  
    corr_ensemble_mid[iensemble] = corr(truth_mid, mod_mid)
    R2_ensemble_mid[iensemble] = L2_R(truth_mid, mod_mid)
    corr_ensemble_bot[iensemble] = corr(truth_bot, mod_bot)
    R2_ensemble_bot[iensemble] = L2_R(truth_bot, mod_bot)

print('full panel, correlation:')
mean_val = np.mean(corr_ensemble_full)
max_val = np.max(corr_ensemble_full)
min_val = np.min(corr_ensemble_full)
std_val = np.std(corr_ensemble_full)
latex_table_row=f', full & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('full panel, R2:')
mean_val = np.mean(R2_ensemble_full)
max_val = np.max(R2_ensemble_full)
min_val = np.min(R2_ensemble_full)
std_val = np.std(R2_ensemble_full)
latex_table_row=f', full & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('midjet panel, correlation:')
mean_val = np.mean(corr_ensemble_mid)
max_val = np.max(corr_ensemble_mid)
min_val = np.min(corr_ensemble_mid)
std_val = np.std(corr_ensemble_mid)
latex_table_row=f', mid & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('midjet panel, R2:')
mean_val = np.mean(R2_ensemble_mid)
max_val = np.max(R2_ensemble_mid)
min_val = np.min(R2_ensemble_mid)
std_val = np.std(R2_ensemble_mid)
latex_table_row=f', mid & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)

mean and variance of all input data:
[ 0.03322287 25.14291526] [0.31186455 0.34117971]
mean and variance of all output data:
[-5.16332561e-04 -9.81347005e-05] [9.36171329e-05 1.01426415e-04]
number of training records (after offset and before padding): 596
number of testing records (after offset): 149
train input shape:
(596, 2, 722, 258)
number of training records (after offset and padding): 640
number of testing records (after offset): 149


/tmp/ipykernel_443280/4197861228.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path)
/tmp/ipykernel_443280/4197861228.py:107: FutureWar

full panel, correlation:
, full & 0.8281 & 0.8367 & 0.8203 & 0.0048 \\
full panel, R2:
, full & 0.6849 & 0.6987 & 0.6723 & 0.0076 \\
midjet panel, correlation:
, mid & 0.7339 & 0.7502 & 0.7215 & 0.0080 \\
midjet panel, R2:
, mid & 0.5384 & 0.5625 & 0.5204 & 0.0118 \\


In [6]:
def apply_fixed_offset_ssh_trails_by_segments(all_input, all_output, shifted_input_indices, seg_lengths, lag=1):
    """
    SSH trails by 1 snapshot relative to selected other input channels, within each segment (.nc file).
    We keep SSH and outputs at time t, but use shifted channels from time t-1.

    Concretely (lag=1):
      x_new[t, all channels] <- all_input[t]  (base is SSH time)
      x_new[t, shifted channels] <- all_input[t-1, shifted channels]
      y_new[t] <- all_output[t]
    To avoid cyclic pairing, we DROP the first snapshot of each segment.

    all_input:  (T, Cin, H, W)
    all_output: (T, Cout, H, W)
    seg_lengths: e.g. [150,150,150,150]
    """
    assert lag == 1, "Written for lag=1 (easy to generalize later)."
    shifted_input_indices = list(shifted_input_indices)

    T, Cin, H, W = all_input.shape
    Tout, Cout = all_output.shape[0], all_output.shape[1]
    assert Tout == T, "Input/output time dimension mismatch."

    new_T = sum(L - lag for L in seg_lengths)  # drop first lag per segment
    x_new = np.zeros((new_T, Cin, H, W), dtype=all_input.dtype)
    y_new = np.zeros((new_T, Cout, H, W), dtype=all_output.dtype)

    in_pos = 0
    out_pos = 0

    for L in seg_lengths:
        if L <= lag:
            raise ValueError(f"Segment length {L} too short for lag={lag}.")

        s0 = in_pos
        s1 = in_pos + L
        keep = L - lag

        # Base alignment is SSH time t = s0+lag ... s1-1 (drop first snapshot)
        x_new[out_pos:out_pos+keep, :, :, :] = all_input[s0+lag:s1, :, :, :]

        # Overwrite shifted channels with t-lag = s0 ... s1-lag-1
        x_new[out_pos:out_pos+keep, shifted_input_indices, :, :] = all_input[s0:s1-lag, shifted_input_indices, :, :]

        # Outputs aligned with SSH time (same as base): drop first snapshot
        y_new[out_pos:out_pos+keep, :, :, :] = all_output[s0+lag:s1, :, :, :]

        in_pos += L
        out_pos += keep

    return x_new, y_new


In [7]:
# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓ Change below for each Configuration ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓
vi1 = 'ssh_ins'
vi2 = 'u_xy_ins'
vi3 = 'v_xy_ins'

vo1 = 'ssh_cos'
vo2 = 'ssh_sin'
#Important: shifted_input_indices a few lines later change in each config. in later codes processing the trained data, remember to use apply_fixed_offset_ssh_trails_by_segments consistent with the lines below
#to make sure that shifted_input_indices are consistent. 

save_fn_prefix  = 'any_{}{}{}_{}{}_nobatchnorm_offset_48hr_ssh_trails'.format(vi1, vi2, vi3, vo1, vo2)

batch_size = 60 #maximizing it so that the GPU memory maxes out. Needs to be divisible by Ntrain. Otherwise there will be size mismatch issues.

var_input_names = [vi1, vi2, vi3]
var_output_names = [vo1, vo2]
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑
N_inp = len(var_input_names)
N_out = len(var_output_names)

nctrains, nctest = hf.load_data_from_nc_as_lists(root_dir)

# Add offset, using the ordering in var_input_names:
# var_input_names = ['ssh_ins','u_xy_ins','v_xy_ins']
# segment lengths per .nc file (train: 4 sims, test: 1 sim)
seg_train = [nc.dimensions['time_counter'].size for nc in nctrains]
seg_test  = [nc.dimensions['time_counter'].size for nc in nctest]

all_train_input, all_train_output = preload_data(nctrains, Ntrain_fromfiles)
all_test_input,  all_test_output  = preload_data(nctest,  Ntest_fromfiles)

# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓ Change below for each Configuration ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓
all_train_input, all_train_output = apply_fixed_offset_ssh_trails_by_segments(
    all_train_input, all_train_output, shifted_input_indices=[1,2], seg_lengths=seg_train, lag=1
)
all_test_input, all_test_output = apply_fixed_offset_ssh_trails_by_segments(
    all_test_input, all_test_output, shifted_input_indices=[1,2], seg_lengths=seg_test, lag=1
)
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑


#Normalize data
#Compute mean and variance for normalization
mean_input=np.nanmean(np.concatenate((all_train_input, all_test_input), axis=0),axis=(0, 2, 3))
mean_output=np.nanmean(np.concatenate((all_train_output, all_test_output), axis=0),axis=(0, 2, 3))
#Subtract the data with their means
all_train_input=all_train_input-mean_input[None, :, None, None]
all_train_output=all_train_output-mean_output[None, :, None, None]
all_test_input=all_test_input-mean_input[None, :, None, None]
all_test_output=all_test_output-mean_output[None, :, None, None]
#Compute the variances
var_input=np.nanmean((np.concatenate((all_train_input, all_test_input), axis=0))**2,axis=(0, 2, 3))
var_output=np.nanmean((np.concatenate((all_train_output, all_test_output), axis=0))**2,axis=(0, 2, 3))
print("mean and variance of all input data:")
print(mean_input,var_input)
print("mean and variance of all output data:")
print(mean_output,var_output)
#Scale the data so that they have variance of 1
all_train_input=all_train_input/np.sqrt(var_input[None, :, None, None])
all_train_output=all_train_output/np.sqrt(var_output[None, :, None, None])
all_test_input=all_test_input/np.sqrt(var_input[None, :, None, None])
all_test_output=all_test_output/np.sqrt(var_output[None, :, None, None])
#Have checked that after these operations, the data is scaled to be zero mean and variance 1.

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and before padding):', Ntrain)
print('number of testing records (after offset):', Ntest)
print('train input shape:')
print(all_train_input.shape)


#Pad so that the number of training data is divisible by batch 
all_train_input, all_train_output = pad_to_multiple_by_repeating_last(
    all_train_input, all_train_output, batch_size)

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and padding):', Ntrain)
print('number of testing records (after offset):', Ntest)


inp_test, out_test = loaddata_preloaded_test(all_test_input, all_test_output)
out_test = out_test*np.sqrt(var_output[None, :, None, None])+mean_output[None, :, None, None]

truth_bot = out_test[:, :, bottom_slice, :]
truth_mid = out_test[:, :, mid_slice, :]
truth_top = out_test[:, :, top_slice, :]

combined_names = ''.join(var_input_names)

#Array to record performance metrics
corr_ensemble_full = np.zeros(nensemble)
R2_ensemble_full = np.zeros(nensemble)
corr_ensemble_top = np.zeros(nensemble)
R2_ensemble_top = np.zeros(nensemble)
corr_ensemble_mid = np.zeros(nensemble)
R2_ensemble_mid = np.zeros(nensemble)
corr_ensemble_bot = np.zeros(nensemble)
R2_ensemble_bot = np.zeros(nensemble)
for iensemble in np.arange(nensemble):
    fstr = f'{save_fn_prefix}_rp_{iensemble}' 
    model_filename = f'/{fstr}.pth'
    model_path = model_folder+ model_filename
    state_dict = torch.load(model_path)
    # Create a new instance of the model
    model = UNet_nobatchnorm(N_inp, N_out, bilinear = True, Nbase = Nbase)
    # Load the state_dict into the model
    model.load_state_dict(state_dict)
    # Set the model to evaluation mode
    model.eval()
    
    with torch.no_grad():
        out_mod = model(totorch(inp_test)).detach().cpu().numpy()
    #Renormalize
    out_mod = out_mod*np.sqrt(var_output[None, :, None, None])+mean_output[None, :, None, None]

    mod_bot = out_mod[:, :, bottom_slice, :]
    mod_mid = out_mod[:, :, mid_slice, :]
    mod_top = out_mod[:, :, top_slice, :]

    corr_ensemble_full[iensemble] = corr(out_test, out_mod)
    R2_ensemble_full[iensemble] = L2_R(out_test, out_mod)    
    corr_ensemble_top[iensemble] = corr(truth_top, mod_top)
    R2_ensemble_top[iensemble] = L2_R(truth_top, mod_top)  
    corr_ensemble_mid[iensemble] = corr(truth_mid, mod_mid)
    R2_ensemble_mid[iensemble] = L2_R(truth_mid, mod_mid)
    corr_ensemble_bot[iensemble] = corr(truth_bot, mod_bot)
    R2_ensemble_bot[iensemble] = L2_R(truth_bot, mod_bot)

print('full panel, correlation:')
mean_val = np.mean(corr_ensemble_full)
max_val = np.max(corr_ensemble_full)
min_val = np.min(corr_ensemble_full)
std_val = np.std(corr_ensemble_full)
latex_table_row=f', full & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('full panel, R2:')
mean_val = np.mean(R2_ensemble_full)
max_val = np.max(R2_ensemble_full)
min_val = np.min(R2_ensemble_full)
std_val = np.std(R2_ensemble_full)
latex_table_row=f', full & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('midjet panel, correlation:')
mean_val = np.mean(corr_ensemble_mid)
max_val = np.max(corr_ensemble_mid)
min_val = np.min(corr_ensemble_mid)
std_val = np.std(corr_ensemble_mid)
latex_table_row=f', mid & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('midjet panel, R2:')
mean_val = np.mean(R2_ensemble_mid)
max_val = np.max(R2_ensemble_mid)
min_val = np.min(R2_ensemble_mid)
std_val = np.std(R2_ensemble_mid)
latex_table_row=f', mid & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)

mean and variance of all input data:
[ 0.03322337  0.03565685 -0.00190692] [0.31172642 0.04678398 0.04873947]
mean and variance of all output data:
[-5.19675373e-04 -9.81703693e-05] [9.42781426e-05 1.02080678e-04]
number of training records (after offset and before padding): 596
number of testing records (after offset): 149
train input shape:
(596, 3, 722, 258)
number of training records (after offset and padding): 600
number of testing records (after offset): 149


/tmp/ipykernel_443280/1155646153.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path)
/tmp/ipykernel_443280/1155646153.py:107: FutureWar

full panel, correlation:
, full & 0.8779 & 0.8904 & 0.8663 & 0.0076 \\
full panel, R2:
, full & 0.7676 & 0.7923 & 0.7400 & 0.0156 \\
midjet panel, correlation:
, mid & 0.8171 & 0.8339 & 0.8038 & 0.0104 \\
midjet panel, R2:
, mid & 0.6608 & 0.6944 & 0.6238 & 0.0212 \\


In [8]:
# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓ Change below for each Configuration ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓
vi1 = 'ssh_ins'
vi2 = 'T_xy_ins'

vo1 = 'ssh_cos'
vo2 = 'ssh_sin'
#Important: shifted_input_indices a few lines later change in each config. in later codes processing the trained data, remember to use apply_fixed_offset_ssh_trails_by_segments consistent with the lines below
#to make sure that shifted_input_indices are consistent. 


save_fn_prefix  = 'any_{}{}_{}{}_nobatchnorm_offset_48hr_ssh_trails'.format(vi1, vi2, vo1, vo2)
var_input_names = [vi1, vi2]
var_output_names = [vo1, vo2]

batch_size = 80 #maximizing it so that the GPU memory maxes out. Needs to be divisible by Ntrain. Otherwise there will be size mismatch issues.

# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑
N_inp = len(var_input_names)
N_out = len(var_output_names)

nctrains, nctest = hf.load_data_from_nc_as_lists(root_dir)

# Add offset, using the ordering in var_input_names:
# var_input_names = ['ssh_ins','u_xy_ins','v_xy_ins']
# segment lengths per .nc file (train: 4 sims, test: 1 sim)
seg_train = [nc.dimensions['time_counter'].size for nc in nctrains]
seg_test  = [nc.dimensions['time_counter'].size for nc in nctest]

all_train_input, all_train_output = preload_data(nctrains, Ntrain_fromfiles)
all_test_input,  all_test_output  = preload_data(nctest,  Ntest_fromfiles)

# ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓ Change below for each Configuration ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

#Add offset
all_train_input, all_train_output = apply_fixed_offset_ssh_trails_by_segments(
    all_train_input, all_train_output, shifted_input_indices=[1], seg_lengths=seg_train, lag=1
)
all_test_input, all_test_output = apply_fixed_offset_ssh_trails_by_segments(
    all_test_input, all_test_output, shifted_input_indices=[1], seg_lengths=seg_test, lag=1
)

# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑


#Normalize data
#Compute mean and variance for normalization
mean_input=np.nanmean(np.concatenate((all_train_input, all_test_input), axis=0),axis=(0, 2, 3))
mean_output=np.nanmean(np.concatenate((all_train_output, all_test_output), axis=0),axis=(0, 2, 3))
#Subtract the data with their means
all_train_input=all_train_input-mean_input[None, :, None, None]
all_train_output=all_train_output-mean_output[None, :, None, None]
all_test_input=all_test_input-mean_input[None, :, None, None]
all_test_output=all_test_output-mean_output[None, :, None, None]
#Compute the variances
var_input=np.nanmean((np.concatenate((all_train_input, all_test_input), axis=0))**2,axis=(0, 2, 3))
var_output=np.nanmean((np.concatenate((all_train_output, all_test_output), axis=0))**2,axis=(0, 2, 3))
print("mean and variance of all input data:")
print(mean_input,var_input)
print("mean and variance of all output data:")
print(mean_output,var_output)
#Scale the data so that they have variance of 1
all_train_input=all_train_input/np.sqrt(var_input[None, :, None, None])
all_train_output=all_train_output/np.sqrt(var_output[None, :, None, None])
all_test_input=all_test_input/np.sqrt(var_input[None, :, None, None])
all_test_output=all_test_output/np.sqrt(var_output[None, :, None, None])
#Have checked that after these operations, the data is scaled to be zero mean and variance 1.

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and before padding):', Ntrain)
print('number of testing records (after offset):', Ntest)
print('train input shape:')
print(all_train_input.shape)


#Pad so that the number of training data is divisible by batch 
all_train_input, all_train_output = pad_to_multiple_by_repeating_last(
    all_train_input, all_train_output, batch_size)

# update record counts 
Ntrain = all_train_input.shape[0]
Ntest  = all_test_input.shape[0]
print('number of training records (after offset and padding):', Ntrain)
print('number of testing records (after offset):', Ntest)


inp_test, out_test = loaddata_preloaded_test(all_test_input, all_test_output)
out_test = out_test*np.sqrt(var_output[None, :, None, None])+mean_output[None, :, None, None]

truth_bot = out_test[:, :, bottom_slice, :]
truth_mid = out_test[:, :, mid_slice, :]
truth_top = out_test[:, :, top_slice, :]

combined_names = ''.join(var_input_names)

#Array to record performance metrics
corr_ensemble_full = np.zeros(nensemble)
R2_ensemble_full = np.zeros(nensemble)
corr_ensemble_top = np.zeros(nensemble)
R2_ensemble_top = np.zeros(nensemble)
corr_ensemble_mid = np.zeros(nensemble)
R2_ensemble_mid = np.zeros(nensemble)
corr_ensemble_bot = np.zeros(nensemble)
R2_ensemble_bot = np.zeros(nensemble)
for iensemble in np.arange(nensemble):
    fstr = f'{save_fn_prefix}_rp_{iensemble}' 
    model_filename = f'/{fstr}.pth'
    model_path = model_folder+ model_filename
    state_dict = torch.load(model_path)
    # Create a new instance of the model
    model = UNet_nobatchnorm(N_inp, N_out, bilinear = True, Nbase = Nbase)
    # Load the state_dict into the model
    model.load_state_dict(state_dict)
    # Set the model to evaluation mode
    model.eval()
    
    with torch.no_grad():
        out_mod = model(totorch(inp_test)).detach().cpu().numpy()
    #Renormalize
    out_mod = out_mod*np.sqrt(var_output[None, :, None, None])+mean_output[None, :, None, None]

    mod_bot = out_mod[:, :, bottom_slice, :]
    mod_mid = out_mod[:, :, mid_slice, :]
    mod_top = out_mod[:, :, top_slice, :]

    corr_ensemble_full[iensemble] = corr(out_test, out_mod)
    R2_ensemble_full[iensemble] = L2_R(out_test, out_mod)    
    corr_ensemble_top[iensemble] = corr(truth_top, mod_top)
    R2_ensemble_top[iensemble] = L2_R(truth_top, mod_top)  
    corr_ensemble_mid[iensemble] = corr(truth_mid, mod_mid)
    R2_ensemble_mid[iensemble] = L2_R(truth_mid, mod_mid)
    corr_ensemble_bot[iensemble] = corr(truth_bot, mod_bot)
    R2_ensemble_bot[iensemble] = L2_R(truth_bot, mod_bot)

print('full panel, correlation:')
mean_val = np.mean(corr_ensemble_full)
max_val = np.max(corr_ensemble_full)
min_val = np.min(corr_ensemble_full)
std_val = np.std(corr_ensemble_full)
latex_table_row=f', full & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('full panel, R2:')
mean_val = np.mean(R2_ensemble_full)
max_val = np.max(R2_ensemble_full)
min_val = np.min(R2_ensemble_full)
std_val = np.std(R2_ensemble_full)
latex_table_row=f', full & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('midjet panel, correlation:')
mean_val = np.mean(corr_ensemble_mid)
max_val = np.max(corr_ensemble_mid)
min_val = np.min(corr_ensemble_mid)
std_val = np.std(corr_ensemble_mid)
latex_table_row=f', mid & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)
print('midjet panel, R2:')
mean_val = np.mean(R2_ensemble_mid)
max_val = np.max(R2_ensemble_mid)
min_val = np.min(R2_ensemble_mid)
std_val = np.std(R2_ensemble_mid)
latex_table_row=f', mid & {mean_val:.4f} & {max_val:.4f} & {min_val:.4f} & {std_val:.4f} \\\\'
print(latex_table_row)

mean and variance of all input data:
[ 0.03322337 25.1429548 ] [0.31172642 0.34121342]
mean and variance of all output data:
[-5.19675373e-04 -9.81703693e-05] [9.42781426e-05 1.02080678e-04]
number of training records (after offset and before padding): 596
number of testing records (after offset): 149
train input shape:
(596, 2, 722, 258)
number of training records (after offset and padding): 640
number of testing records (after offset): 149


/tmp/ipykernel_443280/2832208265.py:110: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path)
/tmp/ipykernel_443280/2832208265.py:110: FutureWar

full panel, correlation:
, full & 0.8290 & 0.8351 & 0.8237 & 0.0038 \\
full panel, R2:
, full & 0.6864 & 0.6966 & 0.6781 & 0.0062 \\
midjet panel, correlation:
, mid & 0.7372 & 0.7454 & 0.7267 & 0.0055 \\
midjet panel, R2:
, mid & 0.5433 & 0.5556 & 0.5280 & 0.0082 \\
